# Lab 2: Pandas for Cat vs Dog Faces

In this notebook, you will treat the cat-and-dog-faces dataset as a **table**, not just a folder of images.

We will use Pandas to:

- inspect metadata
- check class balance
- create useful derived columns
- compare predictions from Lab 1
- analyze where the NumPy baseline fails


Set `STUDENT_ID` in the first code cell. Each notebook uses it as the random seed for sampling, split suggestions, and visualization so every student gets a reproducible variant.

**Questions in this lab**

1. Load and inspect dataset metadata
2. Audit missing values and duplicates
3. Create derived columns for analysis
4. Build a stratified split suggestion
5. Merge NumPy baseline predictions
6. Compute accuracy tables
7. Analyze failure patterns and recommend one improvement


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from lab_utils.visualization import (
    plot_class_balance,
    plot_error_rate_by_group,
    plot_numeric_distribution,
)

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "data" / "cats_dogs_faces_small").exists():
            return candidate
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "cats_dogs_faces_small"
METADATA_PATH = DATA_ROOT / "metadata.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

LABELS = ("cat", "dog")
SPLITS = ("train", "val", "test")
STUDENT_ID = 10422021  # Replace with your own student ID.
SEED = int(STUDENT_ID)
np.random.seed(SEED)
NUMPY_PRED_PATH = ARTIFACT_DIR / f"lab1_numpy_predictions_{STUDENT_ID}.csv"

def build_metadata_from_folders(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for label in LABELS:
            label_dir = data_root / split / label
            for path in sorted(label_dir.glob("*.jpg")) + sorted(label_dir.glob("*.png")):
                with Image.open(path) as image:
                    image = image.convert("RGB")
                    width, height = image.size
                    arr = np.asarray(image, dtype=np.float32) / 255.0
                rows.append(
                    {
                        "filepath": str(path.relative_to(data_root)),
                        "label": label,
                        "split": split,
                        "width": width,
                        "height": height,
                        "mean_intensity": float(arr.mean()),
                    }
                )
    return pd.DataFrame(rows)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the prepared subset at data/cats_dogs_faces_small/."
    )

if METADATA_PATH.exists():
    df = pd.read_csv(METADATA_PATH)
else:
    df = build_metadata_from_folders(DATA_ROOT)
    print("metadata.csv was not found, so a dataframe was built directly from the folders.")

print(df.head())
print(df.shape)


### Visual Helper: Quick Dataset Overview

These plots give students a fast visual summary of class balance and brightness before they start writing Pandas analysis code.


In [ ]:
plot_class_balance(df)

if "mean_intensity" in df.columns:
    plot_numeric_distribution(
        df,
        column="mean_intensity",
        group_col="label",
        title="Mean intensity by label",
    )


## Question 1: What does the dataset table tell us?

Use Pandas operations to answer:

- How many rows are in the dataframe?
- What are the column names?
- How many cats and dogs do we have?
- How are examples distributed across splits?


In [ ]:
# TODO: replace the placeholders with real Pandas operations
num_rows = 0
column_names = []
class_counts = None
split_counts = None

print("Rows:", num_rows)
print("Columns:", column_names)
print("\nClass counts:")
print(class_counts)
print("\nSplit counts:")
print(split_counts)

assert num_rows == len(df), "The row count should match the dataframe length."
assert "label" in column_names and "filepath" in column_names, "Important metadata columns are missing."


## Question 2: Is the metadata clean?

Check for:

- missing values
- duplicated file paths
- unexpected labels
- invalid image sizes


In [ ]:
def audit_metadata(frame: pd.DataFrame) -> dict:
    # TODO: return a dictionary with these keys:
    #   missing_values
    #   duplicate_filepaths
    #   bad_labels
    #   non_positive_sizes
    raise NotImplementedError("Write a small metadata audit function.")

audit_report = audit_metadata(df)
audit_report


## Question 3: Create useful columns for analysis

Add these columns:

- `pixel_count = width * height`
- `aspect_ratio = width / height`
- `brightness_band` using 4 bins from darkest to brightest
- `size_bucket` using labels such as `small`, `medium`, and `large`

Use `pd.qcut` for the brightness bands.


In [ ]:
analysis_df = df.copy()

# TODO: create the requested columns on analysis_df
# Suggested idea for size buckets:
#   pixel_count < 64*64 -> "small"
#   pixel_count == 64*64 -> "medium"
#   pixel_count > 64*64 -> "large"

analysis_df.head()


## Question 4: Can you create a stratified split suggestion?

Pretend the dataset came without a split column. Create a new column called
`recommended_split` with a **70 / 15 / 15** split, while keeping cats and dogs balanced.

Hint:

1. Shuffle the rows once with a fixed random seed
2. Count the rank of each example inside its label group
3. Convert ranks into split labels


In [ ]:
shuffled_df = analysis_df.sample(frac=1.0, random_state=SEED).copy()

# TODO: build a balanced recommended_split column using groupby + cumcount
# The final column should contain only: train, val, test

split_balance = (
    shuffled_df.groupby(["recommended_split", "label"]).size().unstack(fill_value=0)
    if "recommended_split" in shuffled_df.columns
    else None
)
split_balance


## Question 5: Merge the NumPy baseline predictions

Load the CSV saved in Lab 1 and merge it with the metadata dataframe using `filepath`.
Then create a boolean column called `correct_numpy` if it is missing.


In [ ]:
if not NUMPY_PRED_PATH.exists():
    raise FileNotFoundError(
        f"Run Lab 1 first with STUDENT_ID={STUDENT_ID} so {NUMPY_PRED_PATH.name} is available."
    )

numpy_pred_df = pd.read_csv(NUMPY_PRED_PATH)

# TODO: merge analysis_df with numpy_pred_df on filepath
merged_df = None

if merged_df is not None and "correct_numpy" not in merged_df.columns:
    merged_df["correct_numpy"] = merged_df["label"] == merged_df["pred_numpy"]

merged_df.head() if merged_df is not None else "Complete the merge first."


## Question 6: Where is the baseline doing well or poorly?

Build summary tables for:

- overall NumPy accuracy
- NumPy accuracy by split
- NumPy accuracy by class
- count of correct vs incorrect predictions


In [ ]:
if merged_df is None:
    raise ValueError("Complete Question 5 before continuing.")

# TODO: compute the requested summary tables
overall_accuracy = 0.0
accuracy_by_split = None
accuracy_by_class = None
correctness_counts = None

print("Overall accuracy:", overall_accuracy)
print("\nAccuracy by split:")
print(accuracy_by_split)
print("\nAccuracy by class:")
print(accuracy_by_class)
print("\nCorrect vs incorrect:")
print(correctness_counts)

plot_error_rate_by_group(
    merged_df,
    group_col="split",
    correct_col="correct_numpy",
    title="NumPy error rate by split",
)


## Question 7: Analyze failure patterns

Focus only on the mistakes and answer:

- Are errors more common in darker images or brighter images?
- Are errors more common in smaller images?
- Which class is harder for the NumPy baseline?

Save your final merged dataframe to:

`artifacts/lab2_error_analysis.csv`


In [ ]:
if merged_df is None:
    raise ValueError("Complete Question 5 before continuing.")

errors_only = merged_df[~merged_df["correct_numpy"]].copy()

# TODO: create at least two grouped tables that help explain the mistakes
# Example ideas:
#   errors_only.groupby("brightness_band").size()
#   errors_only.groupby(["label", "size_bucket"]).size().unstack(fill_value=0)

output_path = ARTIFACT_DIR / "lab2_error_analysis.csv"
merged_df.to_csv(output_path, index=False)
print(f"Saved merged analysis dataframe to: {output_path}")

if "brightness_band" in merged_df.columns:
    plot_error_rate_by_group(
        merged_df,
        group_col="brightness_band",
        correct_col="correct_numpy",
        title="NumPy error rate by brightness band",
    )

if "size_bucket" in merged_df.columns:
    plot_error_rate_by_group(
        merged_df,
        group_col="size_bucket",
        correct_col="correct_numpy",
        title="NumPy error rate by size bucket",
    )


## Reflection

Write short answers to these questions:

1. Is the dataset balanced across classes and splits?
2. What kind of images seem hardest for the NumPy baseline?
3. What is one data or modeling improvement you would try before Lab 3?

**Optional extension**

Compare the average `mean_intensity` of correct vs incorrect predictions and decide whether brightness looks truly important.
